# Lab 5 Exercise, part 1: Expedia with human approval

This notebook solves the first part of the Lab 5 exercise: sign in to Expedia before searching for flights.

The Lab 5 code already supports this flow:

- `request_human_help` asks the user to take over the browser.
- `HumanInTheLoopMiddleware` pauses before human help or a push notification.
- `Sidekick.resume()` approves the pending action and continues the turn.

The Sidekick opens Expedia and pauses at sign-in. Log in in its browser window, then resume the run. Credentials never appear in this notebook.

Use the lab's Windows fix for the MCP error log. It does nothing on macOS and Linux.

In [ ]:
import sys

if sys.platform == "win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

## Imports

Add the lab folder to the path and import its `Sidekick` class.

In [ ]:
import os
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join("..", "..")))

from dotenv import load_dotenv
import sidekick as sidekick_module
from sidekick import Sidekick

load_dotenv(override=True)

## Set the sandbox

Point the sandbox at this contribution before `setup()`. The Sidekick creates the folder if needed.

In [ ]:
sandbox = os.path.abspath("sandbox")
sidekick_module.SANDBOX = sandbox

sidekick = Sidekick()
await sidekick.setup()
print(f"Sidekick ready with {len(sidekick.tools)} tools")

## The Expedia flight task

Ask the Sidekick to sign in before searching. The task contains no credentials.

The browser opens and the Sidekick should pause at the sign-in page.

In [ ]:
flight_task = """Go to https://www.expedia.com in your browser and sign in to my Expedia account first.
You cannot log in yourself, so once you reach the sign-in page, ask me to do it for you.
Then find me the best round-trip flight from New York to London, leaving about a month from now
and returning a week later. I care about price first, then total journey time, and I would rather avoid
itineraries with two or more stops. Write your recommendation with the top three options to expedia_flights.md,
then send me a push notification with the price of your top pick."""

flight_criteria = """The user was asked to log in to Expedia, expedia_flights.md is written with three
specific options including airline, times and price, plus a clear recommendation, and a push notification
was sent with the recommended price."""

history = await sidekick.run_turn(flight_task, flight_criteria, history=[])
print(history[-1]["content"])

## First pause: your turn at the keyboard

Log in through the Sidekick's browser before running the next cell. Resuming approves the help request.

A captcha or block page may appear. Complete it in the browser, then resume again if asked.

In [ ]:
history = await sidekick.resume(history)
print(history[-1]["content"])

## Approve the push notification

After login, the Sidekick searches, writes `expedia_flights.md`, and asks before sending the push notification.

Approve the pending action. When the run ends, the evaluator's result is the last entry.

In [ ]:
if sidekick.paused:
    history = await sidekick.resume(history)
for entry in history[-2:]:
    print(f"[{entry['role']}] {entry['content']}\n")

## The plan it followed

Print `sidekick.todos` to review its plan.

In [ ]:
for todo in sidekick.todos:
    print(f"[{todo['status']}] {todo['content']}")

## Read the file it wrote

Read the generated flight report.

In [ ]:
print((Path(sandbox) / "expedia_flights.md").read_text(encoding="utf-8"))

## Cleanup

Close the MCP sessions and browser.

In [ ]:
sidekick.cleanup()